# 문서 탐색 (EDA) — 제공 뉴스

벡터 DB 구성 전에 문서의 구성과 품질을 파악한다 (가이드 4-4).
청킹 방식, 전처리 규칙, 품질 필터(`indexer/quality.py`) 기준은 이 노트북에서 확인한 결과를 근거로 정하고 리포트 2장에 기술한다.

**실행 전**: 제공 CSV 를 `data/provided/` 에 넣고 `python -m collector provided` 로 적재한다.

> 출력에는 기사 본문 일부가 포함되므로, 커밋 전에 **Clear All Outputs** 로 지운다.

In [ ]:
import json
import re
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.express as px
from IPython.display import display

from common.docstore import open_db
from common.schema import DocType
# 판정 규칙은 노트북에 다시 쓰지 않고 인덱서 코드에서 가져온다 → EDA 숫자 = 실제 인덱싱 결과
from indexer.quality import (
    CITY_PATTERN, DATELINE_PATTERN, DEFAULT_TOPIC_TERMS, NOT_PLACE, PROVINCE_PATTERNS, SEOUL_PATTERN,
    TITLE_MARKERS, QualityInput, assess, get_quality_config,
)

CONFIG = get_quality_config("v1")

px.defaults.template = "plotly_white"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]  # 범주 색상 고정 순서 (1~3번)

with open_db(read_only=True) as conn:
    news = pd.read_sql_query("SELECT * FROM documents WHERE doc_type = 'news'", conn)

news["extra"] = news["extra"].map(json.loads)
news["published"] = pd.to_datetime(news["published_at"], errors="coerce")
news["text"] = news["raw_text"].fillna("")
news["body_len"] = news["text"].str.len()
news["file_group"] = news["extra"].map(
    lambda e: " + ".join(sorted({"소상공인" if f.startswith("01_") else "지역화폐" for f in e["provided_files"]}))
)
print(f"뉴스 {len(news):,}건")


def show_samples(df, n, seed, width=150):
    """제목과 본문 앞부분을 출력한다. 수치만으로는 문서 성격을 알 수 없으므로 직접 읽는다."""
    for _, r in df.sample(min(n, len(df)), random_state=seed).iterrows():
        print(f"[{r['body_len']:,}자] {r['source']} | {r['title']}\n    {r['text'][:width]}")

## 1. 구성
- 파일(그룹)별, 수집 키워드별 건수
- 매체 분포, 매체명 결손·도메인 형태 복구 여부

In [ ]:
display(news["file_group"].value_counts().rename("건수").to_frame())

# 묶음 파일의 복수 키워드 표기 형식은 원본마다 다를 수 있어 흔한 구분자를 모두 나눈다.
keywords = news["keyword"].fillna("").str.split(r"\s*[;,|/]\s*", regex=True).explode()
display(keywords[keywords != ""].value_counts().rename("건수").to_frame())

missing_source = news["extra"].map(lambda e: e["source_missing"])
print(f"매체 수 {news['source'].nunique():,} / 매체명 결손 {missing_source.mean():.1%}")
if missing_source.any():
    display(news[missing_source].groupby(news["published"].dt.to_period("M")).size().rename("결손 건수").tail(12).to_frame())
# 제공 측에서 URL·도메인으로 매체명을 복구했으므로, 매체명 자리에 도메인이 들어간 경우를 확인한다.
domain_like = news["source"].str.contains(r"\.(?:com|co\.kr|kr|net)\b", regex=True)
print(f"매체명이 도메인 형태인 기사: {domain_like.sum():,}건")
display(news.loc[domain_like, "source"].value_counts().head(10).rename("건수").to_frame())

top_sources = news["source"].value_counts().head(20).rename_axis("매체").reset_index(name="기사 수")
fig = px.bar(top_sources, x="기사 수", y="매체", orientation="h", title="매체별 기사 수 (상위 20)",
             color_discrete_sequence=SERIES[:1])
fig.update_yaxes(categoryorder="total ascending", title=None)
fig.update_layout(bargap=0.35)
fig.show()

## 2. 기간
- 발행일 분포, 특정 시기 편중 여부

In [ ]:
print("기간:", news["published"].min(), "~", news["published"].max())
print("발행일 미상:", news["published"].isna().sum())

monthly = (
    news.dropna(subset=["published"])
    .assign(month=lambda d: d["published"].dt.to_period("M").dt.to_timestamp())
    .groupby(["month", "file_group"]).size().reset_index(name="기사 수")
)
fig = px.line(monthly, x="month", y="기사 수", color="file_group", title="월별 기사 수",
              color_discrete_sequence=SERIES, labels={"month": "", "file_group": "그룹"})
fig.update_traces(line_width=2)
fig.show()

## 3. 길이
- 본문 길이 분포, 과소·과다 문서 비율 → `QualityConfig.min_chars` 후보
- 짧은 문서와 가장 긴 문서를 직접 읽어 성격을 확인한다

In [ ]:
print(news["body_len"].describe(percentiles=[.05, .1, .25, .5, .75, .9, .95, .99]).round(0))
for n in (100, 200, 300, 500):
    short = news["body_len"] < n
    print(f"본문 {n}자 미만: {short.sum():,}건 ({short.mean():.1%})")

q99 = news["body_len"].quantile(0.99)
fig = px.histogram(news[news["body_len"] <= q99], x="body_len", nbins=60, title="본문 길이 분포 (상위 1% 제외)",
                   labels={"body_len": "본문 길이(자)"}, color_discrete_sequence=SERIES[:1])
fig.update_layout(bargap=0.05, yaxis_title="기사 수")
fig.show()

print("--- 300자 미만 샘플: 최소 길이 기준 판단용")
show_samples(news[news["body_len"] < 300], 10, seed=1)
for n in (5_000, 10_000, 20_000):
    print(f"본문 {n:,}자 초과: {(news['body_len'] > n).sum():,}건")
print("\n--- 5,000~10,000자 샘플: 정상 장문 기사와 전문·녹취가 섞이는 구간인지 확인")
show_samples(news[news["body_len"].between(5_000, 10_000)], 10, seed=4, width=100)
print("\n--- 가장 긴 기사 10건: 청크를 과도하게 만드는 문서인지 확인")
display(news.nlargest(10, "body_len")[["body_len", "source", "title"]])

## 4. 품질
- 광고·사진·인사·정기 편성 기사 표지, 보일러플레이트 비중

In [ ]:
# 제목 표지는 indexer/quality.py 의 TITLE_MARKERS (괄호가 닫히는 위치까지 맞춰 연재명 오탐 방지)
BODY_MARKERS = {
    "저작권 문구": r"무단\s*전재|재배포\s*금지|Copyright|ⓒ|©",
    "AI 학습 금지 문구": r"AI\s*학습\s*(?:이용)?\s*금지",
    "기자 이메일": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "특수 줄바꿈 문자(LS·PS)": "[\u2028\u2029]",
}
title_hits = {k: news["title"].str.contains(p, regex=True) for k, (p, _) in TITLE_MARKERS.items()}
rows = [("제목", k, m.sum(), m.mean()) for k, m in title_hits.items()]
body_hits = {k: news["text"].str.contains(p, regex=True) for k, p in BODY_MARKERS.items()}
rows += [("본문", k, m.sum(), m.mean()) for k, m in body_hits.items()]
markers = pd.DataFrame(rows, columns=["위치", "표지", "건수", "비율"])
markers["비율"] = markers["비율"].map("{:.1%}".format)
display(markers)

# 제외 후보: 광고, 사진·영상, 인사·부고·게시판, 전문·녹취, 정기 편성
news["marker_excluded"] = pd.concat(title_hits.values(), axis=1).any(axis=1)
for k, m in title_hits.items():
    print(f"--- {k} 예시")
    print(news.loc[m, "title"].head(5).to_string() if m.any() else "    (없음)")

## 5. 서울 비중
- 통신사 기사의 첫머리 발신지 표기 `(서울=연합뉴스)` 는 기사 내용과 무관하므로 제거한 뒤 판정한다
- '서울'이 들어간 고유명사(서울경제, 서울신문, 서울대 등)와 중복 지명(중구·강서구)도 제외한다
- 키워드 단순 매칭이라 근사치다. 오판은 샘플을 읽어 확인한다

In [ ]:
LEAD_CHARS = CONFIG.lead_chars  # 제목 + 본문 앞부분을 '리드'로 본다
DATELINE, SEOUL = DATELINE_PATTERN, SEOUL_PATTERN

lead_raw = news["title"] + " " + news["text"].str.slice(0, LEAD_CHARS)
lead = lead_raw.str.replace(DATELINE, " ", regex=True)
news["seoul_lead"] = lead.str.contains(SEOUL, regex=True)
print(f"서울 언급 비율  발신지 표기 포함: {lead_raw.str.contains(SEOUL, regex=True).mean():.1%}"
      f"  →  제거 후: {news['seoul_lead'].mean():.1%}")
display(news.groupby("file_group")["seoul_lead"].mean().map("{:.1%}".format).rename("서울 언급 비율").to_frame())

REGIONS = {"서울": SEOUL_PATTERN, **PROVINCE_PATTERNS}
region_hits = {r: lead.str.contains(p, regex=True) for r, p in REGIONS.items()}
regions = pd.Series({r: m.mean() for r, m in region_hits.items()})
fig = px.bar(regions.rename_axis("지역").reset_index(name="비율"), x="비율", y="지역", orientation="h",
             title=f"제목·리드 {LEAD_CHARS}자 내 광역 지자체 언급 비율",
             color_discrete_sequence=SERIES[:1])
fig.update_yaxes(categoryorder="total ascending", title=None)
fig.update_xaxes(tickformat=".0%")
fig.update_layout(bargap=0.35, height=560)
fig.show()

## 6. 범위 분류 — 서울 언급 / 타 지역만 언급 / 지역 언급 없음
- 범위 규칙(정책은 전국, 지자체 사업은 서울)에 따라 '타 지역만 언급'이 제외 후보다

In [ ]:
# 범위 규칙: 정책은 전국 단위, 지자체 사업은 서울. 따라서 '서울 언급'뿐 아니라
# 특정 지역이 없는 전국 단위 기사도 범위 안이다. 타 지역만 언급한 기사가 제외 후보다.
# 광역 지자체명만으로는 '하남시', '청주시' 같은 기초 지자체 기사를 놓친다 (첫 실행 샘플에서 확인).
# 'OO시는', 'OO군이'처럼 지자체가 주어로 나오는 형태를 근사로 잡고, 잡힌 이름을 직접 확인한다.
CITY = CITY_PATTERN  # 'OO시는', 'OO특례시가' 등. NOT_PLACE 로 대도시·신도시 등 제외
city_names = lead.str.findall(CITY).map(lambda names: [n for n in names if n not in NOT_PLACE])
print("--- 탐지된 기초 지자체 상위 30개: 지명이 아닌 단어가 섞였는지 확인 (섞였으면 NOT_PLACE 에 추가)")
print(city_names.explode().value_counts().head(30).to_string())

other_region = pd.concat([m for r, m in region_hits.items() if r != "서울"], axis=1).any(axis=1)
other_region |= city_names.map(bool)
news["scope"] = "지역 언급 없음"
news.loc[other_region, "scope"] = "타 지역만 언급"
news.loc[news["seoul_lead"], "scope"] = "서울 언급"
display(pd.crosstab(news["scope"], news["file_group"], margins=True, margins_name="합계"))

print("--- '지역 언급 없음' 샘플: 전국 정책 기사인지, 지역 기사인데 지명이 리드 밖에 있는지 확인")
show_samples(news[news["scope"] == "지역 언급 없음"], 10, seed=2)

## 7. 주제 관련성
- 제목·리드에 주제어가 없는 기사는 키워드가 본문에만 스친 주제 무관 기사일 가능성이 높다

In [ ]:
# 키워드가 본문 어딘가에만 스쳐 나오는 기사(정치·경제 일반)를 걸러내기 위한 관련성 판정어 후보
TOPIC_TERMS = "|".join(map(re.escape, CONFIG.topic_terms))
news["topic_lead"] = lead.str.contains(TOPIC_TERMS, regex=True)
print(f"제목·리드에 주제어가 있는 기사: {news['topic_lead'].mean():.1%}")
display(news.groupby("file_group")["topic_lead"].mean().map("{:.1%}".format).rename("주제어 포함 비율").to_frame())

print("--- 주제어가 없는 기사 샘플: 주제와 무관한지 확인")
show_samples(news[~news["topic_lead"]], 10, seed=3)

## 8. 중복 현황
- 같은 URL, 본문 완전 일치, 괄호 머리말을 뗀 제목 일치 → `indexer/dedup.py` 기준 근거

In [ ]:
nonempty = news["text"] != ""
news["norm_title"] = (news["title"].str.replace(r"[\[【(][^\]】)]*[\]】)]", "", regex=True)
                      .str.replace(r"\W+", "", regex=True))
titled = news["norm_title"] != ""
dup = pd.DataFrame({
    "관련 문서 수": {
        "같은 URL (발행일만 다름)": news["url"].duplicated(keep=False).sum(),
        "본문 완전 일치": news.loc[nonempty, "text"].duplicated(keep=False).sum(),
        "정규화 제목 일치": news.loc[titled, "norm_title"].duplicated(keep=False).sum(),
    },
    "대표 1건 외 제거 대상": {
        "같은 URL (발행일만 다름)": news["url"].duplicated().sum(),
        "본문 완전 일치": news.loc[nonempty, "text"].duplicated().sum(),
        "정규화 제목 일치": news.loc[titled, "norm_title"].duplicated().sum(),
    },
})
display(dup)

groups = news[titled].groupby("norm_title").filter(lambda g: len(g) > 1)
top = groups["norm_title"].value_counts().head(5).index
display(groups[groups["norm_title"].isin(top)][["norm_title", "published_at", "source", "body_len", "title"]]
        .sort_values(["norm_title", "published_at"]))

## 9. 내용 — 샘플 직접 읽기
- 구조와 서술 방식, 제거할 반복 문구, 핵심 내용의 위치를 기록한다

In [ ]:
SAMPLE_PER_GROUP = 5
for group, g in news.groupby("file_group"):
    print(f"===== {group} =====")
    show_samples(g, SAMPLE_PER_GROUP, seed=42, width=300)

## 10. 부족 자료 탐색 — 제공되지 않은 가이드 키워드의 본문 등장 건수
- 등장이 적은 항목은 직접 수집(보도자료·서울시 자료·보고서·법령)으로 보강하거나 답변 불가 문항으로 쓴다

In [ ]:
GUIDE_TERMS = {
    "소상공인 정책자금": r"정책\s*자금",
    "상권활성화구역": r"상권\s*활성화\s*구역",
    "지역사랑상품권": r"지역사랑\s*상품권|서울사랑\s*상품권",
    "디지털 전환": r"디지털\s*대?전환",
    "상가 공실": r"공실",
    "상권 임대료": r"임대료",
    "젠트리피케이션": r"젠트리피케이션|둥지\s*내몰림",
    "생활인구": r"생활\s*인구",
    "카드·가맹점 수수료": r"(?:카드|가맹점)\s*수수료",
    "소상공인시장진흥공단": r"소상공인시장진흥공단|소진공",
}
in_scope = news["scope"].isin(["서울 언급", "지역 언급 없음"])
coverage = pd.DataFrame({
    "전체": {k: news["text"].str.contains(p, regex=True).sum() for k, p in GUIDE_TERMS.items()},
    "범위 내 (서울+지역 언급 없음)": {k: news.loc[in_scope, "text"].str.contains(p, regex=True).sum() for k, p in GUIDE_TERMS.items()},
    "서울 언급": {k: news.loc[news["seoul_lead"], "text"].str.contains(p, regex=True).sum() for k, p in GUIDE_TERMS.items()},
})
display(coverage)

fig = px.bar(coverage["범위 내 (서울+지역 언급 없음)"].rename_axis("항목").reset_index(name="기사 수"),
             x="기사 수", y="항목", orientation="h", title="범위 내 기사 중 항목별 등장 건수",
             color_discrete_sequence=SERIES[:1])
fig.update_yaxes(categoryorder="total ascending", title=None)
fig.update_layout(bargap=0.35)
fig.show()

## 11. 지식베이스 후보 규모 — `indexer/quality.py` v1 적용 결과
- 인덱서가 쓸 `assess()` 를 그대로 돌린다. 사유는 판정 순서상 처음 걸린 하나만 센다
- 기간 규칙(최근 2년)의 근거가 된 연도별·기간별 표를 함께 남긴다

In [ ]:
from indexer.dedup import content_hash
from indexer.parse import clean_text

# prepare 와 같게 정제 본문(저작권 문구 제거 등)으로 판정한다. 1~10번 탐색은 원문 기준이다.
news["clean"] = news["text"].map(clean_text)
decisions = [
    assess(QualityInput(doc_id=r.doc_id, doc_type=DocType.NEWS, title=r.title, text=r.clean,
                        source=r.source, published_at=r.published_at, extra=r.extra), CONFIG)
    for r in news.itertuples(index=False)
]
news["q_reason"] = [d.reason.value if d.excluded else "유지" for d in decisions]
news["q_scope"] = [d.scope.value if d.scope else None for d in decisions]

ORDER = ["too_short", "too_long", "low_information", "promotional", "transcript",
         "off_topic", "outside_seoul", "out_of_period", "유지"]
display(news["q_reason"].value_counts().reindex(ORDER, fill_value=0).rename("건수").to_frame())

# 남은 문서 안에서만 중복을 판정해야, 앞 단계에서 빠진 원본 때문에 사본까지 사라지지 않는다.
# 완전 중복은 indexer/dedup.py 와 같게 공백·기호를 뗀 정제 본문의 해시로 판정한다.
keep = news["q_reason"] == "유지"
dupes = news.loc[keep, "clean"].map(content_hash).duplicated()
keep[dupes[dupes].index] = False
print(f"품질 필터 통과 {int((news['q_reason'] == '유지').sum()):,}건 → 완전 중복 제거 후 {int(keep.sum()):,}건")
candidates = news[keep]
display(candidates["file_group"].value_counts().rename("최종 후보").to_frame())
display(candidates["q_scope"].value_counts().rename("최종 후보").to_frame())

# 기간 규칙의 근거: 기간 규칙만 빼고 모두 통과한 기사의 연도별·기간별 건수
pool = news[news["q_reason"].isin(["유지", "out_of_period"])]
display(pool.groupby(pool["published"].dt.year).size().rename("기간 규칙 전 후보").to_frame())
latest = news["published"].max()
display(pd.DataFrame(
    {f"최근 {y}년": [int((pool["published"] > latest - pd.DateOffset(years=y)).sum())] for y in (1, 2, 3, 5)},
    index=["기간 규칙 전 후보"],
))

CHUNK_CHARS = 800
total_chars = int(candidates["clean"].str.len().sum())
print(f"최종 후보 본문 합계 {total_chars:,}자 → 청크 약 {total_chars // CHUNK_CHARS:,}개 (청크 {CHUNK_CHARS}자 가정)")
print("※ 재배포본(본문 유사) 처리는 아직 반영되지 않았다. 실제 지식베이스는 이보다 작아진다.")

## 12. 재배포본 — 중복 처리 임계값 (`indexer/dedup.py`)
- 11번 최종 후보끼리 발행일 3일 이내 쌍의 본문 유사도(문자 5-gram Jaccard)를 본다
- 구간별 표본을 읽고 **같은 기사의 재배포**와 **관점이 다른 별개 기사**가 갈리는 지점을 임계값으로 정한다
- 관점이 다른 기사까지 지우면 상충 근거 제시가 불가능해지므로 임계값을 과도하게 낮추지 않는다 (가이드 4-4)

In [ ]:
from indexer.dedup import DedupInput, count_duplicates, get_dedup_config, similar_pairs

DEDUP = get_dedup_config("v1")
docs = [DedupInput(r.doc_id, DocType.NEWS, r.clean, r.published_at) for r in candidates.itertuples(index=False)]
pairs = pd.DataFrame(similar_pairs(docs, DEDUP, min_similarity=0.3), columns=["a", "b", "similarity"])
print(f"비교 대상 {len(docs):,}건 / 유사도 0.3 이상 쌍 {len(pairs):,}개 (발행일 차이 {DEDUP.max_day_gap}일 이내)")

fig = px.histogram(pairs, x="similarity", nbins=35, range_x=[0.3, 1.0], title="문서 쌍 유사도 분포 (문자 5-gram Jaccard)",
                   labels={"similarity": "유사도"}, color_discrete_sequence=SERIES[:1])
fig.update_layout(bargap=0.05, yaxis_title="문서 쌍 수")
fig.show()

info = candidates.set_index("doc_id")
for lo, hi in [(0.3, 0.5), (0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 1.01)]:
    band = pairs[(pairs["similarity"] >= lo) & (pairs["similarity"] < hi)]
    print(f"\n--- 유사도 {lo:.1f}~{min(hi, 1.0):.1f}: {len(band):,}쌍 — 같은 기사의 재배포인가, 별개 기사인가?")
    for _, pair in band.sample(min(5, len(band)), random_state=5).iterrows():
        a, b = info.loc[pair["a"]], info.loc[pair["b"]]
        print(f"[{pair['similarity']:.2f}] {a['published_at'][:10]} {a['source']} | {a['title']}")
        print(f"       {b['published_at'][:10]} {b['source']} | {b['title']}")

rows = []
for threshold in (0.5, 0.6, 0.7, 0.8, 0.9):
    removed = count_duplicates(info.index, pairs.itertuples(index=False, name=None), threshold)
    rows.append((threshold, removed, len(docs) - removed))
display(pd.DataFrame(rows, columns=["임계값", "제거되는 재배포본", "남는 뉴스"]))
print("최종 건수는 python -m indexer prepare 출력이 기준이다 (완전 중복과 재배포본을 함께 묶어 계산).")

## 정리 (`indexer/quality.py` v1 에 반영됨)
- 최소·최대 길이: 100자 / 10,000자 (3번)
- 제목 표지 제외: 사진·영상, 광고, 인사·부고·게시판, 전문·녹취, 정기 편성 (4번)
- 범위: 서울 언급 + 지역 언급 없음(전국 단위) 유지, 타 지역만 언급은 제외 (5·6번)
- 주제 관련성: 제목·리드 300자에 주제어 (7번)
- 기간: 수집 기간 끝 기준 최근 2년 (11번)
- 중복 처리 기준: 발행일 3일 이내 + 본문 유사도 임계값 (12번으로 확정)
- 직접 수집 보강 대상 (10번): 상권활성화구역·생활인구·카드수수료·젠트리피케이션